# 00: Exploration

Checks run against live StatsBomb data during development, and a walk through the cached season review. See README.md for the full write-up.

**Confirmed facts:**
- StatsBomb's exact team name is `"NJ/NY Gotham FC"`, not `"Gotham FC"`
- 137 matches and 3,530 shots in the free 2023 NWSL release; 25 Gotham matches
- Freeze frames on 99% of league shots; 4 of Gotham's 5 penalties have none, one reason penalties are excluded
- No matches outside the 2023 season

In [ ]:
import sys
import warnings

sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

from src.data.statsbomb_loader import get_gotham_matches  # noqa: E402

matches = get_gotham_matches()
print(f"{len(matches)} Gotham matches found")
matches[["match_date", "home_team", "away_team", "home_score", "away_score"]]

## Pipeline

`python -m src.pipeline` (run here or from the repo root) fetches every 2023 NWSL shot once, fits the league xG model on the matches Gotham did not play in, and grades Gotham's shots for and against. Everything is cached to `data/processed/`.

In [ ]:
from src.pipeline import run_full_pipeline

review = run_full_pipeline()["league_review"]
review["holdout_metrics"].round(3)

## Results

- Held out on Gotham (568 shots, 50 goals), the enhanced model predicts 49.8 goals, AUC 0.759.
- Fewest shots conceded per match in the league (9.1), at the highest xG per shot conceded (0.105).
- Goals vs. xG: 26 vs. 26.0 scored, 24 vs. 23.8 conceded. No player outside their plausible range.
- Early season vs. title run: xG conceded per match 1.07 to 0.78 (p = 0.013), suggestive but not clearing the 0.01 threshold.

In [ ]:
review["side_summary"].round(3).T

In [ ]:
review["season_tests"].round(3)

In [ ]:
import matplotlib.pyplot as plt

from src.config import load_config
from src.viz import review_plots

cfg = load_config()
team = cfg["statsbomb"]["team_name"]
w = cfg["season_windows"]
windows = {
    "Early season": (w["early_season"]["start_date"], w["early_season"]["end_date"]),
    "Title run": (w["title_run"]["start_date"], w["title_run"]["end_date"]),
}
review_plots.plot_volume_vs_quality(review["team_table"], team)
review_plots.plot_match_xg(review["match_series"], windows, team)
plt.show()